# Writing constraints as formulas

`mvr_formula.py` turns a string into MVR constraints. It parses the formula and lowers it onto the constructors in `mvr_constraints.py` and the operators in `mvr_operators.py` — it builds no automaton of its own, so every formula is shorthand for calls you could have written by hand.

```
never A                    the path never visits A
reach B before A[2]        B is hit before the 2nd visit to A
count(B) in [2,4]          between two and four B's
reach C between 3 and 6    C is reached within a window
```

Two entry points, both returning a **list** with one entry per top-level `and`, so each feeds a `constraints=` argument directly:

| | returns | for |
| --- | --- | --- |
| `build_mvr(hmm, formula)` | `list[BaseMVR]` | `MVR_CHMM(constraints=...)` |
| `build_mvr_functor(formula)` | `list[MVRConstraint]` | `ConstrainedHiddenMarkovModel(constraints=...)` |

The syntax is summarised in [`FORMULA_CHEATSHEET.md`](FORMULA_CHEATSHEET.md). This notebook uses the same three-state HMM and observation sequence as the other MVR notebooks.

In [ ]:
import itertools
import warnings

import numpy as np
import matplotlib.pyplot as plt
import torch

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.constrained_hmm import ConstrainedHiddenMarkovModel
from conin.hidden_markov_model.mvr_formula import build_mvr, build_mvr_functor
from conin.hidden_markov_model.sampling.ffbs_mvr import ffbs_torch_mvr_chmm


HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={
        "A": 0.2765440507007986,
        "B": 0.4033576072467887,
        "C": 0.32009834205241255,
    },
    transition_probs={
        ("A", "A"): 0.3391777054270445,
        ("A", "B"): 0.049711711669595204,
        ("A", "C"): 0.6111105829033604,
        ("B", "A"): 0.48102507253852517,
        ("B", "B"): 0.05601918704283972,
        ("B", "C"): 0.4629557404186351,
        ("C", "A"): 0.43616112524444134,
        ("C", "B"): 0.1773076392327265,
        ("C", "C"): 0.38653123552283214,
    },
    emission_probs={
        ("A", "lo"): 0.19949219710155375,
        ("A", "mid"): 0.30789837305397333,
        ("A", "hi"): 0.492609429844473,
        ("B", "lo"): 0.534907622618408,
        ("B", "mid"): 0.234417585356662,
        ("B", "hi"): 0.23067479202493,
        ("C", "lo"): 0.09093879934300991,
        ("C", "mid"): 0.008996844382398088,
        ("C", "hi"): 0.9000643562745919,
    },
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)


def run(mvr, path):
    """Evaluate one MVR on a hidden path, by hand."""
    state = mvr.ini[path[0]]

    for h in path[1:]:
        state = mvr.upd[(state, h)]

    return mvr.evl[state]


def holds(mvrs, path):
    """A formula holds iff every constraint it produced holds."""
    return all(run(mvr, path) for mvr in mvrs)


print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## 1. Syntax

`build_mvr` is eager: hand it a model and it returns the MVRs. `build_mvr_functor` is deferred — it returns constraint functors that are called for you at `initialize_chmm` time, which is the path `ConstrainedHiddenMarkovModel` expects.

A formula is checked as it is parsed, so an unknown state or a syntax error is reported against the source text with a caret, not as a failure deep inside a constructor.

In [ ]:
# Eager: straight to MVR objects.
constraints = build_mvr(hmm, "reach B before A[2]")
print("build_mvr         ->", [type(mvr).__name__ for mvr in constraints])
print("mediation states   =", len(constraints[0].mediation_states))

# Deferred: functors, built once the model is known.
chmm = ConstrainedHiddenMarkovModel(hmm=hmm, constraints=build_mvr_functor("reach B before A[2]"))
chmm.initialize_chmm()
print("\nbuild_mvr_functor ->", [c.name for c in chmm.constraints])
print("built              =", [type(mvr).__name__ for mvr in chmm.chmm.constraints])
print("backend            =", chmm.constraint_type)

# Errors point at the offending column.
for bad in ["reach B before A[2", "never Z"]:
    try:
        build_mvr(hmm, bad)
    except Exception as exc:
        print(f"\n{type(exc).__name__}: {exc}")

## 2. What it replaces

Before this layer, expressing "never visit A" meant writing the automaton out: inventing mediation-state names, getting the absorbing update right, and remembering that `evl` is a dict over *mediation* states rather than hidden ones. That function is copy-pasted verbatim into five of the notebooks in this folder.

`never A` is the same automaton. The formula layer is not an approximation of the hand-written version — it lowers to `mvr_not_yet(mvr_current_state(hmm, {"A"}))`, which is precisely how `mvr_forbid_state` is defined.

In [ ]:
# The hand-written version, as it appears in the other notebooks.
mediation_states = ["ok", "violated"]

forbid_A = HomMVR(
    hidden_states=HIDDEN_STATES,
    mediation_states=mediation_states,
    ini={h: ("violated" if h == "A" else "ok") for h in HIDDEN_STATES},
    upd={
        (m, h): ("violated" if m == "violated" or h == "A" else "ok")
        for m in mediation_states
        for h in HIDDEN_STATES
    },
    evl={"ok": True, "violated": False},
)

# The formula.
from_formula = build_mvr(hmm, "never A")

paths = list(itertools.product(HIDDEN_STATES, repeat=4))
agree = all(run(forbid_A, list(p)) == holds(from_formula, list(p)) for p in paths)

print(f"same language on all {len(paths)} paths of length 4: {agree}")
print('one hand-written automaton  ->  "never A"')

## 3. The example from the issue

> *hit B before visiting A twice*

This is a **precedence** between a reachability and a **count** — Dwyer's Precedence-over-Bounded-Existence pattern, and the case that motivates a counting operator: it is expressible in linear temporal logic only as a nested unrolling.

`A[2]` is the count sugar, borrowed from SystemVerilog's goto-repetition: it is the MVR that first becomes true at the 2nd visit to `A`. `before` compares first-satisfaction times, so the whole formula is one `mvr_precedence` call over two operands.

Checked below against an independent reference predicate written directly in Python, by enumerating every path.

In [ ]:
formula = "reach B before A[2]"
constraints = build_mvr(hmm, formula)


def reference(path):
    """Hit B strictly before the 2nd A. Written as the definition, not the recursion."""
    first_B = next((t for t, h in enumerate(path) if h == "B"), None)
    visits_to_A = [t for t, h in enumerate(path) if h == "A"]
    second_A = visits_to_A[1] if len(visits_to_A) > 1 else None

    if first_B is None:
        return False

    return second_A is None or first_B < second_A


def path_weight(path):
    """P(hidden path, observations)."""
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = np.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += np.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in enumerate(observed):
        total += np.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return np.exp(total)


satisfied = evidence = 0.0
disagreements = 0

for path in itertools.product(HIDDEN_STATES, repeat=T):
    path = list(path)
    parsed, expected = holds(constraints, path), reference(path)
    disagreements += parsed != expected

    weight = path_weight(path)
    evidence += weight
    satisfied += weight * parsed

print(f'"{formula}"')
print(f"  lowers to        {len(constraints)} constraint, {len(constraints[0].mediation_states)} mediation states")
print(f"  disagrees with the reference on {disagreements} of {3 ** T} paths")
print(f"  P(satisfied | y) = {satisfied / evidence:.4f}")

## 4. Composition

Three behaviours are worth knowing, because each is a deliberate choice rather than a parser accident.

**A top-level `and` splits into separate constraints** instead of building the product automaton. That is what `mvr_and`'s own warning asks for: downstream algorithms take a list of MVRs and are cheaper for it. An `and` nested under another operator still builds the product, because there it has to.

**Temporal operators chain like Python comparisons**, so `reach A then reach B then reach C` means `(A then B) and (B then C)` — and therefore also splits into two constraints.

**`between a and b` attaches a `time_range`**, the window over which the constraint is enforced. It binds tighter than `and`, so it lands on the nearest expression; parenthesise to window a whole conjunction.

In [ ]:
for formula in [
    "never A and reach B",
    "reach A then reach B then reach C",
    "reach B between 2 and 5",
    "never A and reach B between 2 and 5",
    "(never A and reach B) between 2 and 5",
]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        parsed = build_mvr(hmm, formula)

    windows = [mvr.time_range for mvr in parsed]
    print(f"{formula:<40} -> {len(parsed)} constraint(s), windows {windows}")

## 5. What the constraint does to the posterior

Sampling feasible paths with FFBS and counting how often each hidden state is occupied at each time, against the same model with `constraints=[]`.

The constraint says B must arrive before the second A, so it should pull B *earlier* and push A *later*; C is named nowhere in the formula and should barely move.

In [ ]:
def occupancy(constraints, num_samples=20_000, seed=0):
    """P(hidden = h at time t), estimated from feasible FFBS draws."""
    generator = torch.Generator().manual_seed(seed)
    draws = ffbs_torch_mvr_chmm(
        MVR_CHMM(hidden_markov_model=hmm, constraints=constraints),
        observed,
        num_samples=num_samples,
        generator=generator,
    )

    return {
        h: np.array([np.mean([path[t] == h for path in draws]) for t in range(T)])
        for h in HIDDEN_STATES
    }


free = occupancy([])
held = occupancy(build_mvr(hmm, "reach B before A[2]"))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)

for ax, h in zip(axes, HIDDEN_STATES):
    ax.plot(range(T), free[h], "o--", color="0.6", label="unconstrained")
    ax.plot(range(T), held[h], "o-", color="C0", label="reach B before A[2]")
    ax.set_title(f"P(hidden = {h})")
    ax.set_xlabel("t")
    ax.set_ylim(-0.03, 1.0)

axes[0].set_ylabel("occupancy")
axes[0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

print(f"A at t=0: {free['A'][0]:.3f} -> {held['A'][0]:.3f}")
print(f"B at t=0: {free['B'][0]:.3f} -> {held['B'][0]:.3f}")
print(f"C at t=0: {free['C'][0]:.3f} -> {held['C'][0]:.3f}")

A is pushed out of the early times, B is pulled into them, and C is left alone.

## Where to go next

The formula layer only builds constraints; every algorithm in this folder consumes them. Swap `constraints=[...]` in any other notebook for `build_mvr(hmm, "...")` and it works unchanged.

Anything the grammar cannot say is still reachable by calling `mvr_constraints` and `mvr_operators` directly — the formula layer is a convenience over that algebra, never a replacement for it. The full syntax is in [`FORMULA_CHEATSHEET.md`](FORMULA_CHEATSHEET.md).